# cgoering MP2 - Part 2/3 visualization and gap analysis
Reads `cgoering_project_summary.csv` (Part 1). Monthly series, plots, longest gaps, gap commits, themes, stats.

In [ ]:
import pandas as pd, re, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

NETID = 'cgoering'
SUMMARY = f'{NETID}_project_summary.csv'

# GitHub stars / forks / last commit date, pulled live (Colab has internet; 20 calls is under the free limit)
import requests
REPOS = {
 'gboeing_pynamical':'gboeing/pynamical','mjwoods_rnetcdf':'mjwoods/RNetCDF','r-spatial_spdep':'r-spatial/spdep',
 'svalorzen_ai-toolbox':'Svalorzen/AI-Toolbox','ropensci_datapackager':'ropensci/DataPackageR',
 'facebookresearch_faiss':'facebookresearch/faiss','sasview_sasview':'SasView/sasview',
 'gems-uff_noworkflow':'gems-uff/noworkflow','exascaleinfolab_vadetis':'exascaleinfolab/vadetis','pnnl_isicle':'pnnl/isicle'}
# stars/forks/commit totals read from the GitHub repo pages on 2026-10-08 (used if the API call fails)
FALLBACK = {
 'gboeing_pynamical':[677,116,181],'mjwoods_rnetcdf':[24,10,326],'r-spatial_spdep':[150,28,790],
 'svalorzen_ai-toolbox':[671,102,1751],'ropensci_datapackager':[156,19,686],'facebookresearch_faiss':[40900,4500,2283],
 'sasview_sasview':[78,49,16201],'gems-uff_noworkflow':[126,36,951],'exascaleinfolab_vadetis':[2,2,601],'pnnl_isicle':[64,22,1022]}
GH = {}
for k, repo in REPOS.items():
    try:
        r = requests.get(f'https://api.github.com/repos/{repo}', timeout=20).json()
        c = requests.get(f'https://api.github.com/repos/{repo}/commits?per_page=1', timeout=20).json()
        GH[k] = [r['stargazers_count'], r['forks_count'], c[0]['commit']['committer']['date'][:10].replace('-', '/')]
    except Exception as e:
        print('GitHub API failed for', k, '- using page numbers, LAST COMMIT DATE = WoC max (check by hand):', e)
        GH[k] = [FALLBACK[k][0], FALLBACK[k][1], None]
print(GH)

df = pd.read_csv(SUMMARY, sep=';')
df.columns = ['project_wocid','commit_sha1','author','time','commit message'][:len(df.columns)]
df['time'] = pd.to_numeric(df['time'])
df['dt'] = pd.to_datetime(df['time'], unit='s')
df['commit message'] = df['commit message'].fillna('').astype(str)

THEMES = [
 ('Automated bot contributions', r'dependabot|\bbot\b|pre-commit-ci|renovate|github-actions|\[bot\]|auto-?generated|automated'),
 ('Security patches',            r'security|cve-|vulnerab'),
 ('Release',                     r'\brelease|\bversion bump|bump version|\bv?\d+\.\d+\.\d+|prepare for|\btag\b|cran'),
 ('Dependency updates',          r'\bdepend|requirements|\bbump\b|upgrade|update .*(package|lib|version)|setup\.py|cmake'),
 ('Documentation updates',       r'\bdoc|readme|typo|comment|tutorial|\.md\b|vignette|example'),
 ('Bug fixes',                   r'\bfix|\bbug|\bissue\b|patch|error|crash|resolve|correct'),
 ('Feature development',         r'\badd|feature|implement|\bnew\b|support|enable|introduc|refactor|improv|merge|update'),
]
def theme_of(msg):
    m = msg.lower()
    for name, pat in THEMES:
        if re.search(pat, m): return name
    return 'Other'
def top2(msgs):
    s = pd.Series([theme_of(x) for x in msgs])
    if s.empty: return 'Other:Other'
    t = s.value_counts().index.tolist()
    t = (t + ['Other','Other'])[:2]
    return ':'.join(t)

stats, gap_rows, notes = [], [], {}
for prj, g in df.groupby('project_wocid'):
    g = g.sort_values('time')
    months = g['dt'].dt.to_period('M')
    idx = pd.period_range(months.min(), months.max(), freq='M')
    ts = months.value_counts().reindex(idx, fill_value=0).sort_index()
    ts_df = pd.DataFrame({'Month': ts.index.astype(str), '#Commits': ts.values})
    ts_df.to_csv(f'{NETID}_commits_timeseries_{prj}.csv', sep=';', index=False)
    plt.figure(figsize=(10,5))
    plt.plot(ts_df['Month'], ts_df['#Commits'], marker='o', linestyle='-')
    plt.xlabel('Time (YYYY-MM)'); plt.ylabel('Number of Commits'); plt.title(prj)
    plt.grid(True)
    step = max(1, len(ts_df)//20)
    plt.xticks(range(0, len(ts_df), step), ts_df['Month'][::step], rotation=45)
    plt.tight_layout(); plt.savefig(f'{NETID}_timeseries_{prj}.png', dpi=300); plt.close()

    # gaps = runs of zero-commit months
    runs, start, n = [], None, 0
    for i, v in enumerate(ts.values):
        if v == 0:
            if start is None: start = i
            n += 1
        if (v != 0 or i == len(ts)-1) and start is not None:
            runs.append((start, n)); start, n = None, 0
    ngaps3 = sum(1 for _, l in runs if l >= 3)
    if runs:
        s, l = max(runs, key=lambda r: r[1])
        gstart, gend = str(ts.index[s]), str(ts.index[s+l-1])
        after_idx = ts.index[min(s+l, len(ts)-1)]
        n_after = int(ts[ts.index >= after_idx].sum())
        pre  = g[g['dt'].dt.to_period('M') < ts.index[s]].tail(10)
        post = g[g['dt'].dt.to_period('M') > ts.index[s+l-1]].head(10)
    else:
        gstart = gend = ''; l = 0; n_after = len(g)
        pre = post = g.iloc[0:0]
    for lab, part in (('pre', pre), ('post', post)):
        for _, r in part.iterrows():
            gap_rows.append([lab, prj, r['commit_sha1'], r['author'], r['time'], r['commit message']])

    # activity pattern heuristic
    third = max(1, len(ts)//3)
    a, b, c = ts.iloc[:third].sum(), ts.iloc[third:2*third].sum(), ts.iloc[2*third:].sum()
    if ngaps3 >= 3: pat = 'irregular'
    elif c > 1.5*a and c >= b: pat = 'rising'
    elif a > 1.5*c and a >= b: pat = 'declining'
    elif b < 0.5*min(a, c): pat = 'U-shaped'
    else: pat = 'steady'

    last = g['dt'].max()
    stars, forks, lastgh = GH.get(prj, [0, 0, None])
    if not lastgh: lastgh = last.strftime('%Y/%m/%d')
    status = 'Inactive' if pd.to_datetime(lastgh) < pd.Timestamp('2025-04-01') else 'Active'
    before_t, after_t = top2(pre['commit message']), top2(post['commit message'])
    recent_t = top2(g.tail(10)['commit message'])
    recovered = ('Yes' if len(post) else 'No') if runs and l >= 3 else 'N/A'
    old_a = set(pre['author']); new_a = set(post['author'])
    who = ('Same people' if old_a & new_a else 'New people') if len(post) and len(pre) else 'N/A'
    if not runs or l < 3:
        reason = 'N/A active Project'
    else:
        reason = (f'DRAFT - verify on GitHub: {l}-month gap {gstart} to {gend}; last commits before gap were '
                  f'mostly "{before_t.split(":")[0]}" (e.g. "{pre["commit message"].iloc[-1].strip()[:70] if len(pre) else ""}")')
    why = (f'DRAFT - after gap commits are mostly "{after_t.split(":")[0]}" (e.g. "{post["commit message"].iloc[0].strip()[:70]}")'
           if len(post) else 'N/A')
    note = (f'{prj}: {len(g)} commits, {g["author"].nunique()} authors, pattern {pat}. Longest gap {l} months '
            f'({gstart} to {gend}); {ngaps3} gap(s) of 3+ months. Before gap: {before_t}; after: {after_t}. Status: {status}.')
    with open(f'{NETID}_reflection_{prj}.md', 'w') as f:
        f.write(f'# {prj}\n\n{note}\n\nGap interpretation: {reason}\n\nRecovery: {recovered} ({who}). {why}\n')
    stats.append({'Project': prj, 'ncommits': len(g), 'nauthors': g['author'].nunique(),
        'from': int(g['time'].min()), 'to': int(g['time'].max()), 'nstars': stars, 'nforks': forks,
        'lastGHCommitDate': lastgh, 'LongestGapStart (YYYY-MM)': gstart, 'LongestGapEnd (YYYY-MM)': gend,
        'LongestGapLength': l, 'NumCommitsAfterLastGap': n_after, 'ActivityPattern': pat,
        'NumberOfGapsInTimeline': ngaps3, 'BeforeThemes': before_t, 'AfterThemes': after_t,
        'HypothesizedGapReason': reason, 'HasRecovered': recovered, 'WhyRecovered': why,
        'WhoRecovered': who, 'Currentstatus': status, 'RecentThemes': recent_t, 'Notes': note})

pd.DataFrame(gap_rows, columns=['pre/post','prj','commit','author','time','message']).to_csv(
    f'{NETID}_project_gap_commits.csv', sep=';', index=False)
out = pd.DataFrame(stats); out.to_csv(f'{NETID}_project_stats.csv', sep=';', index=False)
out



In [ ]:
# ---------- paste-ready text for the notebook ----------
print('\n=== PART 1 TEXT CELL (paste into a Markdown cell) ===\n')
print('| project | WoC commits | WoC authors | min time | max time | GH stars | GH forks | GH last commit | GH commit total |')
print('|---|---|---|---|---|---|---|---|---|')
for _, r in out.iterrows():
    f = FALLBACK.get(r['Project'], [None, None, None])
    print(f"| {r['Project']} | {r['ncommits']} | {r['nauthors']} | {pd.to_datetime(r['from'], unit='s'):%Y-%m-%d} | "
          f"{pd.to_datetime(r['to'], unit='s'):%Y-%m-%d} | {r['nstars']} | {r['nforks']} | {r['lastGHCommitDate']} | {f[2]} |")
print('\nValidation: WoC data ends Jan 2025, so GitHub totals can be higher; WoC also dedups forks/merges differently.')
for _, r in out.iterrows():
    gtot = FALLBACK.get(r['Project'], [0, 0, 0])[2]
    if gtot: print(f"  {r['Project']}: WoC {r['ncommits']} vs GitHub {gtot} ({100*r['ncommits']/gtot:.0f}% of GitHub)")

print('\n=== INTERPRETATION CELL DRAFT (paste into a Markdown cell, then edit) ===\n')
print('## Interpretation\n')
for _, r in out.iterrows():
    print(f"- **{r['Project']}** - {r['ActivityPattern']}; longest gap {r['LongestGapLength']} months "
          f"({r['LongestGapStart (YYYY-MM)']} to {r['LongestGapEnd (YYYY-MM)']}); "
          f"{r['NumberOfGapsInTimeline']} gap(s) of 3+ months; status {r['Currentstatus']}.")

import subprocess
subprocess.run(f'zip -q {NETID}_mp2_outputs.zip {NETID}_* ', shell=True)
print(f'\nWrote {NETID}_mp2_outputs.zip - download it from the Files sidebar.')
